# 09. Point processes inferred by SVI (BSTPP)

**Question.** Fit the three competitors of the SP mixture on each dataset, from the data
and without heuristics.

**Data.** The nine datasets of `synthesis_utils.DATASETS` (those whose files are present).
**Outputs.** Fitted models in `out/bstpp_results/` (or `out/bstpp_reinferred/` when
`out/bstpp_results` is a symbolic link, so that linked files are never overwritten).
**Paper.** Competitors of Sec. 3.3, used in Table 2 through `scripts/compute_all.py`.
**Code.** `src/synthesis_utils.py`, BSTPP (Manring et al. 2025, JAX and NumPyro).

| model | intensity |
|---|---|
| LGCP | λ(s) = exp(a₀ + f_s(s)), f_s a Gaussian process |
| Hawkes | λ(t, s) = μ + Σ α f(t − tᵢ) φ(s − sᵢ) |
| Cox-Hawkes | Gaussian-process background plus self-excitation |

BSTPP models are spatio-temporal and our data are not. Times T ~ U(0, T_max) are
assigned, so the temporal parameters are nuisance parameters. The SVI uses at most
`BSTPP_CAP = 2000` points per dataset (memory grows as N²), with 5 000 steps per model.
The full run takes about one hour. Existing models are reloaded unless
`FORCE_FIT = True`.

> **Status (14 November 2026).** Run again with the common chain (BSTPP 0.1.4 with
> JAX 0.9.2 and NumPyro 0.20.1). The 27 models of `out/bstpp_results` are reloaded, none
> was fitted again, so only the 2PCF, γ and JSD columns are new.

## Conventions of the common chain (2PCF, fit window, amplitude ratio, SP)

Since the discussion with Charles (November 2026) every notebook uses one chain,
the one of the calibration (`src/calibration.py`, `src/sp_helpers.py`).

**2PCF.** `methods_two_point_correlation.PCF_with_variance`, called here through
`synthesis_utils.compute_2pcf`. Distances in metres, 20 logarithmic edges from the mean
nearest-neighbour distance of the point set to r_max of the dataset, ξ attached to the
upper edge of the bin, Landy-Szalay with split random catalogues (5 × 7 000 points drawn
inside the border), k = 3 repetitions whose spread gives the error bars.

**Fit window and minimum number of pairs.** The power law ξ(r) = (r/r0)^(−γ) is fitted
on the bins 1 to 18 (bin 0 lies below the mean nearest-neighbour distance, the last one
touches r_max), weighted by the spread over the repetitions. One more rule removes
small-r bins: starting from the smallest r, a bin is dropped as long as it holds fewer
than `MIN_PAIRS = 100` data pairs (DD). The first bin with DD ≥ 100 opens the window and
all the bins above it are kept, so the rule only moves the lower bound. Reason: the
relative Poisson noise of a bin is 1/√DD (10 % at 100 pairs). Below that, ξ of the bin
is set by a handful of pairs and can even be negative, and such bins sit at the small r
where the fit has its lever arm on γ. The window actually used is printed by
`su.describe_fit_window(pcf)`.

**Amplitude ratio.** log a = ⟨log ξ_data − log ξ_SP⟩ over the bins of the same window
where both are positive, then n_rand = m_SP (a^(−1/2) − 1). The mean of logs is
symmetric in data and model and is not driven by one bin where ξ_SP is close to zero,
which the former ρ = ⟨ξ_data/ξ_SP⟩ was. γ and a are measured on the same bins.

**SP generation.** `sp_helpers.soneira_peebles_nd` with `dim = 2`: a child that falls
outside the border is drawn again alone (up to 64 times), so the tree keeps its η^L
leaves, is not truncated, and the generation ends whatever the size of the territory.


In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from shapely.geometry import box as _shapely_box

import jax
import numpyro
import numpyro.distributions as dist
from bstpp.main import LGCP_Model, Hawkes_Model

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT / "src"))
import synthesis_utils as su

DATASET_NAMES = list(su.DATASETS)      # datasets whose files are missing are skipped
FORCE_FIT = False                      # True: fit again even if a .pkl exists
BSTPP_LR = 0.01
BSTPP_NUM_STEPS = 5000                 # 10 000+ for a more precise inference
N_SIM = 5                              # simulations per model for the 2PCF
SEED = su.SEED
T_MAX = 365.0                          # artificial time window (one year)
# Domain [0, 1]^2 as a GeoDataFrame. Passing a numpy array instead triggers a bug
# in BSTPP's _sim_cox (self-join of the computation grid, intensities x5, OOM).
A_GDF = gpd.GeoDataFrame(geometry=[_shapely_box(0.0, 0.0, 1.0, 1.0)])

LOAD_DIR = su.BSTPP_RESULTS
SAVE_DIR = (ROOT / "out" / "bstpp_reinferred") if LOAD_DIR.is_symlink() else LOAD_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = ROOT / "out" / "bstpp"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11,
                     "axes.labelsize": 10, "font.size": 9})
print(f"JAX {jax.__version__} | NumPyro {numpyro.__version__}")
print(f"SVI steps={BSTPP_NUM_STEPS}  lr={BSTPP_LR}  cap={su.BSTPP_CAP}")
print(f"models read from {LOAD_DIR.relative_to(ROOT)}, new fits saved to "
      f"{SAVE_DIR.relative_to(ROOT)}")

JAX 0.9.2 | NumPyro 0.20.1
SVI steps=5000  lr=0.01  cap=2000
models read from out/bstpp_results, new fits saved to out/bstpp_reinferred


## Data

Coordinates normalised to [0, 1]². The training sample (at most 2 000 points, seed 123)
is rebuilt by `su._bstpp_xy`, the same one used by `su.load_bstpp_models`.

In [2]:
loaded = {}
for name in DATASET_NAMES:
    ds = su.DATASETS[name]
    if not (Path(ds["path_border"]).exists() and Path(ds["path_csv"]).exists()):
        print(f"[{name}] skipped, data files not found")
        continue
    print(f"\n--- {name}")
    loaded[name] = su.load_dataset(ds)


--- Austria Cities
  N=2,253  scale=572.9 km


  fit window: bins 1..18 (3.95-235.9 km), 0 small-r bin(s) dropped (DD < 100)
  gamma_data=0.759  r0=1.9 km

--- Belgium Cities
  N=815  scale=272.8 km


  fit window: bins 1..18 (4.42-161.8 km), 0 small-r bin(s) dropped (DD < 100)
  gamma_data=0.731  r0=2.3 km

--- RTE France
  N=4,932  scale=974.0 km


  fit window: bins 1..18 (4.47-131.2 km), 0 small-r bin(s) dropped (DD < 100)
  gamma_data=0.919  r0=12.5 km

--- Gares France
  N=3,470  scale=974.0 km


  fit window: bins 1..18 (5.42-132.6 km), 0 small-r bin(s) dropped (DD < 100)
  gamma_data=0.695  r0=53.3 km

--- ENEDIS France


  N_full=912,982 -> subsampled to 10,000
  N=10,000  scale=974.0 km


... (3 outputs left out)

  fit window: bins 1..18 (4.06-130.5 km), 0 small-r bin(s) dropped (DD < 100)
  gamma_data=0.408  r0=1.0 km

--- Germany Stops


  N_full=660,298 -> subsampled to 10,000
  N=10,000  scale=850.4 km


  fit window: bins 1..18 (3.05-128.4 km), 0 small-r bin(s) dropped (DD < 100)
  gamma_data=0.645  r0=3.5 km

--- Swiss Stops
  N_full=93,288 -> subsampled to 10,000
  N=10,000  scale=346.5 km


  fit window: bins 1..18 (0.65-117.8 km), 0 small-r bin(s) dropped (DD < 100)
  gamma_data=0.632  r0=9.1 km


## Inference

SVI with `BSTPP_NUM_STEPS` steps, posterior samples in `model.samples`.

In [3]:
def training_frame(ld, seed=42):
    """BSTPP data frame of the training sample, sorted by the artificial time T."""
    xy = su._bstpp_xy(ld)
    T = np.random.default_rng(seed).uniform(0, T_MAX, len(xy))
    return (pd.DataFrame({"X": xy[:, 0], "Y": xy[:, 1], "T": T})
            .sort_values("T").reset_index(drop=True))


def new_model(mname, df):
    """Untrained BSTPP model with the priors used for the paper."""
    if mname == "LGCP":
        return LGCP_Model(df, A_GDF, T_MAX, a_0=dist.Normal(0, 5))
    return Hawkes_Model(df, A_GDF, T_MAX, cox_background=(mname == "Cox Hawkes"),
                        a_0=dist.Normal(0, 5), alpha=dist.Beta(2, 5),
                        beta=dist.HalfNormal(2.0), sigmax_2=dist.HalfNormal(0.25))


def pkl_name(name, mname):
    return f"{name.replace(' ', '_')}_{mname.replace(' ', '_')}.pkl"


bstpp_models = {}
for name, ld in loaded.items():
    print(f"\n{'=' * 55}\n  {name}\n{'=' * 55}")
    df = training_frame(ld)
    bstpp_models[name] = {}
    for mname in su.BSTPP_NAMES:
        pkl_in = LOAD_DIR / pkl_name(name, mname)
        try:
            model = new_model(mname, df)
            if pkl_in.exists() and not FORCE_FIT:
                model.load_rslts(str(pkl_in))
                print(f"  [{mname}] reloaded from {pkl_in.name}")
            else:
                print(f"  [{mname}] SVI, {BSTPP_NUM_STEPS} steps ...")
                model.run_svi(lr=BSTPP_LR, num_steps=BSTPP_NUM_STEPS, plot_loss=False)
                model.save_rslts(str(SAVE_DIR / pkl_name(name, mname)))
                print(f"  [{mname}] fitted, saved to {SAVE_DIR.name}/")
            bstpp_models[name][mname] = model
        except Exception as e:
            print(f"  [{mname}] FAILED: {e}")
            bstpp_models[name][mname] = None

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.



  Austria Cities
  [LGCP] reloaded from Austria_Cities_LGCP.pkl
  [Hawkes] reloaded from Austria_Cities_Hawkes.pkl
  [Cox Hawkes] reloaded from Austria_Cities_Cox_Hawkes.pkl

  Belgium Cities


  [LGCP] reloaded from Belgium_Cities_LGCP.pkl
  [Hawkes] reloaded from Belgium_Cities_Hawkes.pkl
  [Cox Hawkes] reloaded from Belgium_Cities_Cox_Hawkes.pkl

  RTE France
  [LGCP] reloaded from RTE_France_LGCP.pkl
  [Hawkes] reloaded from RTE_France_Hawkes.pkl
  [Cox Hawkes] reloaded from RTE_France_Cox_Hawkes.pkl

  Gares France
  [LGCP] reloaded from Gares_France_LGCP.pkl


  [Hawkes] reloaded from Gares_France_Hawkes.pkl
  [Cox Hawkes] reloaded from Gares_France_Cox_Hawkes.pkl

  ENEDIS France
  [LGCP] reloaded from ENEDIS_France_LGCP.pkl
  [Hawkes] reloaded from ENEDIS_France_Hawkes.pkl
  [Cox Hawkes] reloaded from ENEDIS_France_Cox_Hawkes.pkl

  France Communes
  [LGCP] reloaded from France_Communes_LGCP.pkl
  [Hawkes] reloaded from France_Communes_Hawkes.pkl
  [Cox Hawkes] reloaded from France_Communes_Cox_Hawkes.pkl

  France Routes
  [LGCP] reloaded from France_Routes_LGCP.pkl


  [Hawkes] reloaded from France_Routes_Hawkes.pkl
  [Cox Hawkes] reloaded from France_Routes_Cox_Hawkes.pkl

  Germany Stops
  [LGCP] reloaded from Germany_Stops_LGCP.pkl
  [Hawkes] reloaded from Germany_Stops_Hawkes.pkl
  [Cox Hawkes] reloaded from Germany_Stops_Cox_Hawkes.pkl

  Swiss Stops
  [LGCP] reloaded from Swiss_Stops_LGCP.pkl
  [Hawkes] reloaded from Swiss_Stops_Hawkes.pkl
  [Cox Hawkes] reloaded from Swiss_Stops_Cox_Hawkes.pkl


## Posterior summary

In [4]:
post_rows = []
for name, models in bstpp_models.items():
    for mname, model in models.items():
        if model is None:
            continue
        for k in ("a_0", "alpha", "beta", "sigmax_2"):
            if k in model.samples:
                v = np.asarray(model.samples[k])
                if v.ndim == 1:
                    post_rows.append(dict(dataset=name, model=mname, parameter=k,
                                          mean=v.mean(), std=v.std(),
                                          q05=np.percentile(v, 5),
                                          q95=np.percentile(v, 95)))
posterior = pd.DataFrame(post_rows)
posterior.round(4)

,dataset,model,parameter,mean,std,q05,q95
0,Austria Cities,LGCP,a_0,1.1304,0.0838,0.997400,1.271700
1,Austria Cities,Hawkes,a_0,-0.3912,0.1794,-0.676000,-0.088800
2,Austria Cities,Hawkes,alpha,0.9717,0.0037,0.965200,0.977600
3,Austria Cities,Hawkes,beta,4.3297,0.2155,3.978800,4.691200
4,Austria Cities,Hawkes,sigmax_2,0.0006,0.0000,0.000500,0.000600
...,...,...,...,...,...,...,...
76,Swiss Stops,Hawkes,sigmax_2,0.0001,0.0000,0.000100,0.000100
77,Swiss Stops,Cox Hawkes,a_0,0.4696,0.2062,0.142200,0.817200
78,Swiss Stops,Cox Hawkes,alpha,0.9297,0.0102,0.912500,0.944900
79,Swiss Stops,Cox Hawkes,beta,17.4879,0.7126,16.338699,18.676001


## Simulation and 2PCF

`N_SIM` simulations per model, restricted to the border and resampled to
N = min(N_real, 2000), the size of the training sample. The SP mixture and the data are
taken at the same N.

In [5]:
def model_2pcf(model, ld, N, n_sim=N_SIM, seed=SEED):
    """Mean 2PCF over n_sim realisations, and one realisation for the maps."""
    r_ref, xis, xy_ex = None, [], None
    for k in range(n_sim):
        xy = su._bstpp_realization(model, ld["geom_norm"], N, seed + k)
        if xy is None:
            continue
        r, xi = su.compute_2pcf(xy, ld)              # common chain, r in metres
        if r_ref is None:
            r_ref, xy_ex = r, xy
        xis.append(su._align_xi(r, xi, r_ref))
    if r_ref is None:
        return None, None, None
    return r_ref, np.nanmean(xis, axis=0), xy_ex


results = {}
for name, ld in loaded.items():
    ds = su.DATASETS[name]
    N = min(ld["N_real"], su.BSTPP_CAP)
    print(f"\n--- {name}  (N = {N})")
    xy_d, pcf_d = su.data_2pcf_at_N(ld, N)
    r_d, xi_d = pcf_d["r"], pcf_d["xi"]
    print("  data 2PCF:", su.describe_fit_window(pcf_d))
    xy_sp = su.build_sp_mixture(ds, ld, pcf_d, N)
    r_sp, xi_sp = su.compute_2pcf(xy_sp, ld)
    res = {"data": (r_d, xi_d, xy_d), "SP": (r_sp, xi_sp, xy_sp)}
    for mname, model in bstpp_models[name].items():
        res[mname] = model_2pcf(model, ld, N) if model is not None \
            else (None, None, None)
        print(f"  [{mname}] {'ok' if res[mname][0] is not None else 'no realisation'}")
    results[name] = res


--- Austria Cities  (N = 2000)


  data 2PCF: fit window: bins 1..18 (4.19-236.6 km), 0 small-r bin(s) dropped (DD < 100)


  [LGCP] ok


  [Hawkes] ok


  [Cox Hawkes] ok

--- Belgium Cities  (N = 815)
  data 2PCF: fit window: bins 1..18 (4.42-161.8 km), 0 small-r bin(s) dropped (DD < 100)


  [LGCP] ok


... (26 outputs left out)

  data 2PCF: fit window: bins 1..18 (1.87-125.0 km), 0 small-r bin(s) dropped (DD < 100)


  [LGCP] ok


  [Hawkes] ok


  [Cox Hawkes] ok


## Maps and 2PCF

In [6]:
ORDER = ["data", "SP", "LGCP", "Hawkes", "Cox Hawkes"]
COLORS = {"data": "black", "SP": "#2ca02c", "LGCP": "#1f77b4", "Hawkes": "#ff7f0e",
          "Cox Hawkes": "#d62728"}
STYLES = {"data": dict(lw=3.0, ls="-", zorder=10), "SP": dict(lw=2.5, ls="--", zorder=9),
          "LGCP": dict(lw=2.0, ls="-.", zorder=7),
          "Hawkes": dict(lw=2.0, ls=(0, (3, 1, 1, 1)), zorder=7),
          "Cox Hawkes": dict(lw=2.0, ls=":", zorder=8)}


def plot_boundary(ax, geom, color="#444444", lw=0.8):
    for g in (geom.geoms if hasattr(geom, "geoms") else [geom]):
        x, y = g.exterior.xy
        ax.plot(x, y, color=color, lw=lw, zorder=0)


rows = []
for name, res in results.items():
    ds, ld = su.DATASETS[name], loaded[name]
    r_d, xi_d, _ = res["data"]
    jsd = {m: (su.js_divergence(r_d, xi_d, res[m][0], res[m][1])
               if res[m][0] is not None else np.nan) for m in ORDER[1:]}

    fig = plt.figure(figsize=(20, 9))
    gs = gridspec.GridSpec(2, 5, figure=fig, hspace=0.35, wspace=0.25)
    for col, m in enumerate(ORDER):
        ax = fig.add_subplot(gs[0, col])
        xy = res[m][2]
        if xy is not None and len(xy):
            ax.scatter(xy[:, 0], xy[:, 1], s=max(0.5, 3.5 - len(xy) / 2500), alpha=0.35,
                       color=COLORS[m], rasterized=True)
        else:
            ax.text(0.5, 0.5, "no realisation", ha="center", va="center",
                    transform=ax.transAxes, color="gray")
        plot_boundary(ax, ld["geom_norm"])
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        sub = f"\nJSD={jsd[m]:.4f}" if m in jsd and np.isfinite(jsd[m]) else ""
        ax.set_title(m + sub, color=COLORS[m], fontweight="bold", fontsize=9)

    ax = fig.add_subplot(gs[1, :])
    for m in ORDER:
        r, xi, _ = res[m]
        if r is None:
            continue
        ok = np.isfinite(xi) & (r > 0)
        lab = m if m == "data" else f"{m}  (JSD={jsd[m]:.4f})"
        ax.plot(r[ok] / 1e3, xi[ok], color=COLORS[m], label=lab,
                **STYLES[m])
    ax.axhline(0, color="gray", lw=0.7, ls="--")
    ax.set_xscale("log"); ax.set_ylim(bottom=-0.5)
    ax.set_xlabel("r [km]"); ax.set_ylabel(r"$\hat{\xi}(r)$  (Landy-Szalay)")
    ax.set_title(f"2PCF, {name} (γ_data = {ds['gamma_data']:.3f})")
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    fig.suptitle(f"SP against BSTPP, {name} (N = {len(res['data'][2]):,})",
                 fontsize=13, fontweight="bold")
    fig.savefig(FIG_DIR / f"comparison_{name.replace(' ', '_')}_BSTPP.pdf",
                bbox_inches="tight", dpi=150)
    plt.show()

    for m in ORDER[1:]:
        r, xi, _ = res[m]
        g, _ = su.fit_gamma(r, xi) if r is not None else (None, None)
        aic = np.nan
        if m in bstpp_models[name] and bstpp_models[name][m] is not None:
            try:
                aic = float(bstpp_models[name][m].expected_AIC())
            except Exception:
                pass
        rows.append(dict(dataset=name, model=m, gamma=g,
                         dgamma=(g - ds["gamma_data"]) if g is not None else np.nan,
                         jsd=jsd[m], aic=aic))

<Figure size 2200x990 with 6 Axes>

Power-law fit failed: fewer than 3 usable bins.


<Figure size 2200x990 with 6 Axes>

<Figure size 2200x990 with 6 Axes>

<Figure size 2200x990 with 6 Axes>

<Figure size 2200x990 with 6 Axes>

<Figure size 2200x990 with 6 Axes>

<Figure size 2200x990 with 6 Axes>

<Figure size 2200x990 with 6 Axes>

<Figure size 2200x990 with 6 Axes>

## Summary table: γ, JSD and AIC

In [7]:
table = pd.DataFrame(rows)
table.to_csv(ROOT / "out" / "09_bstpp_comparison.csv", index=False)
print(table.round(4).to_string(index=False))
print("\nJSD rank per dataset (1 = best)")
for name, sub in table.dropna(subset=["jsd"]).groupby("dataset", sort=False):
    order = sub.sort_values("jsd")
    print(f"  {name}: " + ", ".join(f"{i}. {m} ({j:.4f})" for i, (m, j) in
                                   enumerate(zip(order.model, order.jsd), 1)))

        dataset      model  gamma  dgamma    jsd         aic
 Austria Cities         SP 0.9103  0.1213 0.0004         NaN
 Austria Cities       LGCP 0.2980 -0.4910 0.0003 -14794.1719
 Austria Cities     Hawkes    NaN     NaN 0.0009 -15589.6855
 Austria Cities Cox Hawkes 0.4127 -0.3763 0.0024 -16067.2930
 Belgium Cities         SP 1.1717  0.3637 0.0019         NaN
 Belgium Cities       LGCP 0.4643 -0.3437 0.0006  -4171.1147
 Belgium Cities     Hawkes 0.2001 -0.6079 0.0044  -4114.8867
 Belgium Cities Cox Hawkes 0.5828 -0.2252 0.0009  -4290.7666
     RTE France         SP 1.0176  0.0866 0.0008         NaN
     RTE France       LGCP 0.3464 -0.5846 0.0045 -12066.4863
     RTE France     Hawkes 1.9500  1.0190 0.0010 -12785.5869
     RTE France Cox Hawkes 1.1940  0.2630 0.0009 -13405.4746
   Gares France         SP 0.9468  0.3038 0.0009         NaN
   Gares France       LGCP 0.1985 -0.4445 0.0173 -12803.3896
   Gares France     Hawkes 1.3786  0.7356 0.0038 -13887.1260
   Gares France Cox Hawk

## Results

- γ of the SP mixture is steeper than γ_cal on every dataset, by 0.09 (RTE) to 0.42
  (municipalities), on the training sample of at most 2 000 points.
- The LGCP is always too flat (γ 0.17 to 0.46, i.e. 0.08 to 0.72 below γ_cal). Its
  Gaussian process has a finite range, see the notes below.
- The Hawkes fit is unstable: γ hits the bound 1.95 on RTE, ENEDIS and the
  municipalities, and the power-law fit fails on the Austrian cities.
- Lowest 2PCF JSD at N = min(N_real, 2 000): LGCP on five datasets (Austrian and Belgian
  cities, municipalities, roads, German stops), SP on two (RTE, railway stations),
  Cox-Hawkes on two (ENEDIS, Swiss stops). The ranking of Table 2 is in notebook 07, at
  N = 3 000 and over 10 realisations.
- Cox-Hawkes has the lowest expected AIC on all nine datasets.

## Notes

- The parameters come with a posterior distribution, and the expected AIC compares the
  models on their likelihood.
- The temporal parameters (β, f_t) cannot be interpreted on spatial data.
- The Gaussian process of LGCP and Cox-Hawkes has a finite range (its VAE is pre-trained
  on a fixed Matérn kernel), which can prevent a power-law 2PCF over several decades.
- Several minutes of SVI per model and dataset, against a fraction of a second for the
  SP calibration (notebook 08).